# 🔍 Notebook 04: Evaluation & GradCAM Explainability
## Test Set Evaluation + Visual Explanations

**Objectives:**
1. ✅ Evaluate trained model on sealed test set
2. ✅ Implement GradCAM for visual explanations
3. ✅ Implement Test-Time Augmentation (TTA)
4. ✅ Generate GradCAM visualizations for all grades
5. ✅ Create evaluation report

**Why GradCAM?**
- Shows which regions the model focuses on
- Helps doctors understand predictions
- Validates model is looking at clinically relevant areas (hemorrhages, exudates)

**Expected Output:**
- `results/test_metrics.json` (sealed test evaluation)
- `artifacts/gradcam_examples.png` (visualization grid)
- `artifacts/tta_comparison.png` (TTA stability analysis)
- `src/xai/gradcam.py` (reusable module)

---

## 1. Setup & Imports

In [ ]:
# Core imports
import os
import sys
import json
from pathlib import Path
from datetime import datetime

# Deep Learning
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Model
import timm

# Image processing
import cv2
import numpy as np
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Data & Metrics
import pandas as pd
from sklearn.metrics import (
    cohen_kappa_score, accuracy_score, confusion_matrix,
    classification_report, precision_recall_fscore_support
)

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Progress bar
from tqdm.notebook import tqdm

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"✅ Using device: {device}")

In [ ]:
# Define paths
PROJECT_ROOT = Path("/Users/shivasaivemula/ALP Project/deep-retina-grade")
DATA_ROOT = PROJECT_ROOT / "aptos2019-blindness-detection"

# Directories
TRAIN_IMAGES_DIR = DATA_ROOT / "train_images"
SPLITS_DIR = PROJECT_ROOT / "splits"
MODELS_DIR = PROJECT_ROOT / "models"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
RESULTS_DIR = PROJECT_ROOT / "results"
SRC_DIR = PROJECT_ROOT / "src"

# Add src to path
sys.path.insert(0, str(SRC_DIR))

# Load splits
df_train = pd.read_csv(SPLITS_DIR / "train.csv")
df_val = pd.read_csv(SPLITS_DIR / "val.csv")
df_test = pd.read_csv(SPLITS_DIR / "test.csv")

print(f"📁 Project Root: {PROJECT_ROOT}")
print(f"📊 Test samples: {len(df_test):,}")

## 2. Load Trained Model

In [ ]:
from models.efficientnet import RetinaModel
from preprocessing.preprocess import RetinaPreprocessor

# Load model - Using EfficientNet-B0 (optimized for M1 Mac)
model = RetinaModel(num_classes=5, pretrained=False, backbone='efficientnet_b0')
checkpoint = torch.load(MODELS_DIR / 'efficientnet_b0_best.pth', map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(device)
model.eval()

print(f"✅ Model loaded from epoch {checkpoint['epoch']}")
print(f"   Best Kappa: {checkpoint['best_kappa']:.4f}")
print(f"   Backbone: EfficientNet-B0")

## 3. Create Test DataLoader

In [ ]:
# ImageNet normalization
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

class RetinaDataset(Dataset):
    """Dataset for evaluation (same as training but simpler)."""
    
    def __init__(self, df, images_dir, img_size=224, transform=None):
        self.df = df.reset_index(drop=True)
        self.images_dir = Path(images_dir)
        self.transform = transform
        self.preprocessor = RetinaPreprocessor(img_size=img_size)
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_id = row['id_code']
        label = row['diagnosis']
        
        img_path = self.images_dir / f"{img_id}.png"
        img = self.preprocessor.preprocess(img_path)
        img = (img * 255).astype(np.uint8)
        
        if self.transform:
            img = self.transform(image=img)['image']
        
        return img, torch.tensor(label, dtype=torch.long), img_id

# Test transforms (no augmentation)
test_transform = A.Compose([
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

# Create test dataset and loader
test_dataset = RetinaDataset(df_test, TRAIN_IMAGES_DIR, transform=test_transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

print(f"✅ Test DataLoader created: {len(test_loader)} batches")

## 4. Evaluate on Sealed Test Set

In [ ]:
def evaluate_model(model, dataloader, device):
    """Comprehensive model evaluation."""
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []
    all_ids = []
    
    with torch.no_grad():
        for images, labels, ids in tqdm(dataloader, desc="Evaluating"):
            images = images.to(device)
            outputs = model(images)
            probs = F.softmax(outputs, dim=1)
            preds = outputs.argmax(dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
            all_probs.extend(probs.cpu().numpy())
            all_ids.extend(ids)
    
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)
    
    # Calculate metrics
    accuracy = accuracy_score(all_labels, all_preds)
    kappa = cohen_kappa_score(all_labels, all_preds, weights='quadratic')
    conf_matrix = confusion_matrix(all_labels, all_preds)
    
    # Per-class metrics
    precision, recall, f1, support = precision_recall_fscore_support(
        all_labels, all_preds, average=None
    )
    
    # Sensitivity for severe DR (Grade 3-4)
    severe_mask = all_labels >= 3
    if severe_mask.sum() > 0:
        severe_sensitivity = (all_preds[severe_mask] >= 3).mean()
    else:
        severe_sensitivity = 0.0
    
    return {
        'accuracy': accuracy,
        'kappa': kappa,
        'confusion_matrix': conf_matrix,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'support': support,
        'severe_sensitivity': severe_sensitivity,
        'predictions': all_preds,
        'labels': all_labels,
        'probabilities': all_probs,
        'ids': all_ids
    }

# Evaluate on test set
test_results = evaluate_model(model, test_loader, device)

print("\n" + "="*70)
print("📊 SEALED TEST SET EVALUATION")
print("="*70)
print(f"Accuracy: {test_results['accuracy']:.4f} ({test_results['accuracy']*100:.1f}%)")
print(f"Quadratic Weighted Kappa: {test_results['kappa']:.4f}")
print(f"Severe DR Sensitivity (Grade 3-4): {test_results['severe_sensitivity']:.4f}")
print(f"\nPer-class Recall (Sensitivity):")
for i, r in enumerate(test_results['recall']):
    print(f"  Grade {i}: {r:.4f} (n={test_results['support'][i]})")
print("="*70)

In [ ]:
# Save test metrics
test_metrics = {
    'accuracy': float(test_results['accuracy']),
    'quadratic_weighted_kappa': float(test_results['kappa']),
    'severe_sensitivity': float(test_results['severe_sensitivity']),
    'per_class_precision': test_results['precision'].tolist(),
    'per_class_recall': test_results['recall'].tolist(),
    'per_class_f1': test_results['f1'].tolist(),
    'per_class_support': test_results['support'].tolist(),
    'confusion_matrix': test_results['confusion_matrix'].tolist(),
    'evaluation_date': datetime.now().isoformat(),
    'model_checkpoint': 'efficientnet_b0_best.pth',
    'test_samples': len(df_test)
}

with open(RESULTS_DIR / 'test_metrics.json', 'w') as f:
    json.dump(test_metrics, f, indent=2)

print(f"✅ Saved: {RESULTS_DIR / 'test_metrics.json'}")

In [ ]:
# Plot confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Raw counts
sns.heatmap(
    test_results['confusion_matrix'], 
    annot=True, 
    fmt='d',
    cmap='Blues',
    xticklabels=['Grade 0', 'Grade 1', 'Grade 2', 'Grade 3', 'Grade 4'],
    yticklabels=['Grade 0', 'Grade 1', 'Grade 2', 'Grade 3', 'Grade 4'],
    ax=axes[0]
)
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_title('Confusion Matrix (Counts)', fontweight='bold')

# Normalized
cm_norm = test_results['confusion_matrix'].astype('float') / test_results['confusion_matrix'].sum(axis=1)[:, np.newaxis]
sns.heatmap(
    cm_norm, 
    annot=True, 
    fmt='.2%',
    cmap='Blues',
    xticklabels=['Grade 0', 'Grade 1', 'Grade 2', 'Grade 3', 'Grade 4'],
    yticklabels=['Grade 0', 'Grade 1', 'Grade 2', 'Grade 3', 'Grade 4'],
    ax=axes[1]
)
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')
axes[1].set_title('Confusion Matrix (Normalized)', fontweight='bold')

plt.suptitle(f'Test Set Evaluation\nAccuracy: {test_results["accuracy"]:.1%} | Kappa: {test_results["kappa"]:.4f}', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'confusion_matrix_test.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✅ Saved: {ARTIFACTS_DIR / 'confusion_matrix_test.png'}")

## 5. Implement GradCAM

GradCAM (Gradient-weighted Class Activation Mapping) generates visual explanations by:
1. Computing gradients of target class w.r.t. final convolutional layer
2. Global-average-pooling gradients to get importance weights
3. Weighted combination of feature maps

In [ ]:
class GradCAM:
    """
    GradCAM implementation for visual explanations.
    
    Reference: Selvaraju et al., "Grad-CAM: Visual Explanations from Deep Networks"
    """
    
    def __init__(self, model, target_layer):
        """
        Args:
            model: The neural network model
            target_layer: The layer to compute GradCAM for
        """
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        
        # Register hooks
        self.target_layer.register_forward_hook(self._forward_hook)
        self.target_layer.register_full_backward_hook(self._backward_hook)
    
    def _forward_hook(self, module, input, output):
        """Store activations during forward pass."""
        self.activations = output.detach()
    
    def _backward_hook(self, module, grad_input, grad_output):
        """Store gradients during backward pass."""
        self.gradients = grad_output[0].detach()
    
    def generate(self, input_tensor, target_class=None):
        """
        Generate GradCAM heatmap.
        
        Args:
            input_tensor: Input image tensor [1, 3, H, W]
            target_class: Target class index (if None, uses predicted class)
            
        Returns:
            heatmap: GradCAM heatmap [H, W] in range [0, 1]
            pred_class: Predicted class
            confidence: Prediction confidence
        """
        self.model.eval()
        
        # Forward pass
        output = self.model(input_tensor)
        pred_class = output.argmax(dim=1).item()
        confidence = F.softmax(output, dim=1)[0, pred_class].item()
        
        # Use predicted class if target not specified
        if target_class is None:
            target_class = pred_class
        
        # Backward pass
        self.model.zero_grad()
        output[0, target_class].backward()
        
        # Get weights (global average pooling of gradients)
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)  # [1, C, 1, 1]
        
        # Weighted combination of activations
        cam = (weights * self.activations).sum(dim=1, keepdim=True)  # [1, 1, H, W]
        
        # ReLU and normalize
        cam = F.relu(cam)
        cam = cam.squeeze().cpu().numpy()
        
        # Resize to input size
        cam = cv2.resize(cam, (input_tensor.shape[3], input_tensor.shape[2]))
        
        # Normalize to [0, 1]
        if cam.max() > 0:
            cam = (cam - cam.min()) / (cam.max() - cam.min())
        
        return cam, pred_class, confidence

print("✅ GradCAM class implemented!")

In [ ]:
def overlay_heatmap(image, heatmap, alpha=0.5, colormap=cv2.COLORMAP_JET):
    """
    Overlay heatmap on image.
    
    Args:
        image: Original image [H, W, 3] in range [0, 1]
        heatmap: Heatmap [H, W] in range [0, 1]
        alpha: Blending factor
        colormap: OpenCV colormap
        
    Returns:
        Blended image [H, W, 3] in range [0, 1]
    """
    # Convert heatmap to color
    heatmap_colored = cv2.applyColorMap((heatmap * 255).astype(np.uint8), colormap)
    heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB) / 255.0
    
    # Blend
    overlaid = alpha * heatmap_colored + (1 - alpha) * image
    overlaid = np.clip(overlaid, 0, 1)
    
    return overlaid

def denormalize(tensor, mean=IMAGENET_MEAN, std=IMAGENET_STD):
    """Denormalize image tensor for visualization."""
    img = tensor.clone()
    for c in range(3):
        img[c] = img[c] * std[c] + mean[c]
    return img.clamp(0, 1).permute(1, 2, 0).numpy()

print("✅ Helper functions defined!")

In [ ]:
# Get target layer for GradCAM (last conv layer of EfficientNet)
# For EfficientNet-B3, we target the last block's output
target_layer = model.backbone.conv_head

# Create GradCAM instance
gradcam = GradCAM(model, target_layer)

print(f"✅ GradCAM initialized with target layer: {type(target_layer).__name__}")

## 6. Generate GradCAM Visualizations

In [ ]:
# Select samples from each grade for visualization
samples_per_grade = []
for grade in range(5):
    grade_samples = df_test[df_test['diagnosis'] == grade]
    if len(grade_samples) > 0:
        sample = grade_samples.sample(1, random_state=42).iloc[0]
        samples_per_grade.append({
            'id_code': sample['id_code'],
            'diagnosis': sample['diagnosis']
        })

print("📊 Selected samples for GradCAM:")
for s in samples_per_grade:
    print(f"   Grade {s['diagnosis']}: {s['id_code']}")

In [ ]:
# Generate GradCAM for each sample
preprocessor = RetinaPreprocessor(img_size=224)

fig, axes = plt.subplots(5, 3, figsize=(15, 25))

for row_idx, sample in enumerate(samples_per_grade):
    img_path = TRAIN_IMAGES_DIR / f"{sample['id_code']}.png"
    
    # Preprocess
    img = preprocessor.preprocess(img_path)
    img_uint8 = (img * 255).astype(np.uint8)
    
    # Apply transform
    transformed = test_transform(image=img_uint8)
    img_tensor = transformed['image'].unsqueeze(0).to(device)
    
    # Generate GradCAM
    heatmap, pred_class, confidence = gradcam.generate(img_tensor)
    
    # Denormalize for visualization
    img_vis = denormalize(img_tensor.squeeze().cpu())
    
    # Overlay heatmap
    overlaid = overlay_heatmap(img_vis, heatmap, alpha=0.5)
    
    # Plot: Original | Heatmap | Overlay
    axes[row_idx, 0].imshow(img_vis)
    axes[row_idx, 0].set_title(f"Original\nActual: Grade {sample['diagnosis']}", fontsize=11)
    axes[row_idx, 0].axis('off')
    
    axes[row_idx, 1].imshow(heatmap, cmap='jet')
    axes[row_idx, 1].set_title(f"GradCAM Heatmap", fontsize=11)
    axes[row_idx, 1].axis('off')
    
    axes[row_idx, 2].imshow(overlaid)
    correct = "✓" if pred_class == sample['diagnosis'] else "✗"
    axes[row_idx, 2].set_title(f"Overlay\nPred: Grade {pred_class} ({confidence:.1%}) {correct}", fontsize=11)
    axes[row_idx, 2].axis('off')

plt.suptitle('GradCAM Explainability - Model Focus Areas by DR Grade', 
             fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'gradcam_examples.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✅ Saved: {ARTIFACTS_DIR / 'gradcam_examples.png'}")

## 7. Implement Test-Time Augmentation (TTA)

TTA improves prediction stability by:
1. Creating multiple augmented versions of input
2. Running inference on each
3. Averaging predictions

This reduces variance for borderline cases (e.g., Grade 2 vs Grade 3).

In [ ]:
class TTAPredictor:
    """
    Test-Time Augmentation for more stable predictions.
    
    Augmentations:
    1. Original
    2. Horizontal flip
    3. Rotate -15°
    4. Rotate +15°
    5. Zoom (1.1x)
    """
    
    def __init__(self, model, device, img_size=224):
        self.model = model
        self.device = device
        self.img_size = img_size
        
        # Base transform (normalization)
        self.normalize = A.Compose([
            A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ToTensorV2(),
        ])
        
        # TTA transforms
        self.tta_transforms = [
            A.Compose([]),  # Original
            A.Compose([A.HorizontalFlip(p=1.0)]),  # H-flip
            A.Compose([A.Rotate(limit=(-15, -15), p=1.0, border_mode=cv2.BORDER_CONSTANT)]),  # Rotate -15
            A.Compose([A.Rotate(limit=(15, 15), p=1.0, border_mode=cv2.BORDER_CONSTANT)]),  # Rotate +15
            A.Compose([A.Affine(scale=1.1, p=1.0)]),  # Zoom 1.1x
        ]
    
    def predict(self, image):
        """
        Predict with TTA.
        
        Args:
            image: Input image (numpy array, uint8, [H, W, 3])
            
        Returns:
            Dictionary with predictions and statistics
        """
        self.model.eval()
        all_probs = []
        
        with torch.no_grad():
            for tta_transform in self.tta_transforms:
                # Apply TTA transform
                augmented = tta_transform(image=image)['image']
                
                # Apply normalization
                normalized = self.normalize(image=augmented)['image']
                
                # Predict
                input_tensor = normalized.unsqueeze(0).to(self.device)
                output = self.model(input_tensor)
                probs = F.softmax(output, dim=1).cpu().numpy()[0]
                all_probs.append(probs)
        
        all_probs = np.array(all_probs)  # [5, num_classes]
        
        # Average predictions
        mean_probs = all_probs.mean(axis=0)
        std_probs = all_probs.std(axis=0)
        
        # Final prediction
        pred_class = mean_probs.argmax()
        confidence = mean_probs[pred_class]
        uncertainty = std_probs[pred_class]
        
        return {
            'prediction': int(pred_class),
            'confidence': float(confidence),
            'uncertainty': float(uncertainty),
            'mean_probs': mean_probs,
            'individual_preds': all_probs.argmax(axis=1),
            'agreement': (all_probs.argmax(axis=1) == pred_class).mean()
        }

# Create TTA predictor
tta_predictor = TTAPredictor(model, device)

print("✅ TTA Predictor created!")
print("   Augmentations: Original, H-Flip, Rotate -15°, Rotate +15°, Zoom 1.1x")

In [ ]:
# Compare single prediction vs TTA on samples
print("\n📊 TTA vs Single Prediction Comparison:")
print("="*70)

comparison_results = []

for sample in samples_per_grade:
    img_path = TRAIN_IMAGES_DIR / f"{sample['id_code']}.png"
    
    # Preprocess
    img = preprocessor.preprocess(img_path)
    img_uint8 = (img * 255).astype(np.uint8)
    
    # Single prediction
    single_input = test_transform(image=img_uint8)['image'].unsqueeze(0).to(device)
    with torch.no_grad():
        single_output = model(single_input)
        single_probs = F.softmax(single_output, dim=1).cpu().numpy()[0]
        single_pred = single_probs.argmax()
        single_conf = single_probs[single_pred]
    
    # TTA prediction
    tta_result = tta_predictor.predict(img_uint8)
    
    print(f"\nGrade {sample['diagnosis']} ({sample['id_code']}):")
    print(f"  Single: Grade {single_pred} ({single_conf:.1%})")
    print(f"  TTA:    Grade {tta_result['prediction']} ({tta_result['confidence']:.1%}) ± {tta_result['uncertainty']:.3f}")
    print(f"  Agreement: {tta_result['agreement']:.0%} ({tta_result['individual_preds']})")
    
    comparison_results.append({
        'id': sample['id_code'],
        'actual': sample['diagnosis'],
        'single_pred': int(single_pred),
        'single_conf': float(single_conf),
        'tta_pred': tta_result['prediction'],
        'tta_conf': tta_result['confidence'],
        'tta_uncertainty': tta_result['uncertainty'],
        'agreement': tta_result['agreement']
    })

print("\n" + "="*70)

In [ ]:
# Visualize TTA results
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Extract data
df_comparison = pd.DataFrame(comparison_results)

# Plot 1: Confidence comparison
x = np.arange(len(df_comparison))
width = 0.35
axes[0].bar(x - width/2, df_comparison['single_conf'], width, label='Single', color='steelblue')
axes[0].bar(x + width/2, df_comparison['tta_conf'], width, label='TTA', color='coral')
axes[0].set_xlabel('Sample')
axes[0].set_ylabel('Confidence')
axes[0].set_title('Prediction Confidence: Single vs TTA', fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels([f"G{r['actual']}" for _, r in df_comparison.iterrows()])
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Plot 2: Uncertainty
axes[1].bar(x, df_comparison['tta_uncertainty'], color='orange')
axes[1].axhline(y=0.1, color='red', linestyle='--', label='Threshold')
axes[1].set_xlabel('Sample')
axes[1].set_ylabel('Uncertainty (Std)')
axes[1].set_title('TTA Uncertainty', fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels([f"G{r['actual']}" for _, r in df_comparison.iterrows()])
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

# Plot 3: Agreement
axes[2].bar(x, df_comparison['agreement'], color='green')
axes[2].axhline(y=0.8, color='red', linestyle='--', label='Threshold (80%)')
axes[2].set_xlabel('Sample')
axes[2].set_ylabel('Agreement (%)')
axes[2].set_title('TTA Prediction Agreement', fontweight='bold')
axes[2].set_xticks(x)
axes[2].set_xticklabels([f"G{r['actual']}" for _, r in df_comparison.iterrows()])
axes[2].legend()
axes[2].grid(axis='y', alpha=0.3)

plt.suptitle('Test-Time Augmentation (TTA) Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'tta_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✅ Saved: {ARTIFACTS_DIR / 'tta_comparison.png'}")

## 8. Save GradCAM Module

In [ ]:
# Save GradCAM as reusable module
gradcam_code = '''
"""
GradCAM - Gradient-weighted Class Activation Mapping

Visual explanations for CNN-based predictions.

Reference: Selvaraju et al., "Grad-CAM: Visual Explanations from Deep Networks
via Gradient-based Localization" (ICCV 2017)

Author: Deep Retina Grade Project
Date: January 2026
"""

import torch
import torch.nn.functional as F
import numpy as np
import cv2
from typing import Optional, Tuple


class GradCAM:
    """
    GradCAM implementation for visual explanations.
    
    This class computes gradient-weighted class activation maps,
    highlighting regions important for the model's prediction.
    
    Usage:
        gradcam = GradCAM(model, target_layer)
        heatmap, pred_class, confidence = gradcam.generate(input_tensor)
    """
    
    def __init__(self, model: torch.nn.Module, target_layer: torch.nn.Module):
        """
        Initialize GradCAM.
        
        Args:
            model: The neural network model
            target_layer: The convolutional layer to compute GradCAM for
        """
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        
        # Register hooks
        self.target_layer.register_forward_hook(self._forward_hook)
        self.target_layer.register_full_backward_hook(self._backward_hook)
    
    def _forward_hook(self, module, input, output):
        """Store activations during forward pass."""
        self.activations = output.detach()
    
    def _backward_hook(self, module, grad_input, grad_output):
        """Store gradients during backward pass."""
        self.gradients = grad_output[0].detach()
    
    def generate(
        self, 
        input_tensor: torch.Tensor, 
        target_class: Optional[int] = None
    ) -> Tuple[np.ndarray, int, float]:
        """
        Generate GradCAM heatmap.
        
        Args:
            input_tensor: Input image tensor [1, 3, H, W]
            target_class: Target class index (if None, uses predicted class)
            
        Returns:
            heatmap: GradCAM heatmap [H, W] in range [0, 1]
            pred_class: Predicted class
            confidence: Prediction confidence
        """
        self.model.eval()
        
        # Forward pass
        output = self.model(input_tensor)
        pred_class = output.argmax(dim=1).item()
        confidence = F.softmax(output, dim=1)[0, pred_class].item()
        
        # Use predicted class if target not specified
        if target_class is None:
            target_class = pred_class
        
        # Backward pass
        self.model.zero_grad()
        output[0, target_class].backward()
        
        # Get weights (global average pooling of gradients)
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        
        # Weighted combination of activations
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        
        # ReLU and normalize
        cam = F.relu(cam)
        cam = cam.squeeze().cpu().numpy()
        
        # Resize to input size
        cam = cv2.resize(cam, (input_tensor.shape[3], input_tensor.shape[2]))
        
        # Normalize to [0, 1]
        if cam.max() > 0:
            cam = (cam - cam.min()) / (cam.max() - cam.min())
        
        return cam, pred_class, confidence


def overlay_heatmap(
    image: np.ndarray, 
    heatmap: np.ndarray, 
    alpha: float = 0.5,
    colormap: int = cv2.COLORMAP_JET
) -> np.ndarray:
    """
    Overlay heatmap on image.
    
    Args:
        image: Original image [H, W, 3] in range [0, 1]
        heatmap: Heatmap [H, W] in range [0, 1]
        alpha: Blending factor (0 = only image, 1 = only heatmap)
        colormap: OpenCV colormap
        
    Returns:
        Blended image [H, W, 3] in range [0, 1]
    """
    # Convert heatmap to color
    heatmap_colored = cv2.applyColorMap(
        (heatmap * 255).astype(np.uint8), 
        colormap
    )
    heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB) / 255.0
    
    # Blend
    overlaid = alpha * heatmap_colored + (1 - alpha) * image
    overlaid = np.clip(overlaid, 0, 1)
    
    return overlaid
'''

# Save to file
gradcam_file = SRC_DIR / "xai" / "gradcam.py"
with open(gradcam_file, 'w') as f:
    f.write(gradcam_code.strip())

print(f"✅ Saved: {gradcam_file}")

## 9. Summary

In [ ]:
print("="*70)
print("📊 EVALUATION & GRADCAM COMPLETE - SUMMARY")
print("="*70)
print(f"""
✅ Test Set Evaluation:
   • Accuracy: {test_results['accuracy']:.1%}
   • Quadratic Weighted Kappa: {test_results['kappa']:.4f}
   • Severe DR Sensitivity: {test_results['severe_sensitivity']:.1%}

✅ GradCAM Explainability:
   • Model: EfficientNet-B0 (optimized for M1)
   • Target layer: conv_head
   • Visualizations generated for all 5 grades
   • Saved as reusable module

✅ Test-Time Augmentation (TTA):
   • 5 augmentations: Original, H-Flip, Rotate ±15°, Zoom
   • Reduces prediction variance for borderline cases
   • Provides uncertainty estimates

📁 Artifacts Generated:
   • {RESULTS_DIR / 'test_metrics.json'}
   • {ARTIFACTS_DIR / 'confusion_matrix_test.png'}
   • {ARTIFACTS_DIR / 'gradcam_examples.png'}
   • {ARTIFACTS_DIR / 'tta_comparison.png'}
   • {SRC_DIR / 'xai' / 'gradcam.py'}
""")
print("="*70)
print("\n🚀 NEXT STEPS (Notebook 05):")
print("   1. Implement Integrated Gradients")
print("   2. Implement LIME")
print("   3. Create Triple XAI comparison")
print("="*70)